# CYGNSS Streamflow Modeling for Ungauged Basins (CSMUB)

### About

This code combines the delineation of watersheds and the analysis of the CYGNSS watermask files to determine the surface water extent of each basin on a monthly time scale using the functions `crop_dem`, `process_basin` and `waterpx_count`. In addition, the function `read_gauge` loads the empirical streamflow data from text files downloaded from the [Global Runoff Data Centre (GRDC)](https://portal.grdc.bafg.de/applications/public.html?publicuser=PublicUser#dataDownload/Home). Lastly, it outputs a .csv file with the station name and the results of the analysis which will be used as the training dataset for the neural network.

---


Author(s): Anna Valcarcel, Samantha Smith

Date of Creation: May 30, 2024

Last Updated: November 5, 2025

Version History:
*   v0.00 - Initial Version taken from demo codes
*   v0.01 - Added data visualization and functionality for reading GRDC data
*   v0.02 - Added .txt file readability for station date, functionality for downloading graphics/plots, and location details within plots.

*   v1.00 - Added functionality for loading station masterfile, cropping continent-scale DEM files, and calculating intersection of delineated and GRDC shape files to determine which shp to use
*   v1.10 - Added functionality for determining landcover classification percentage

*   v2.00 - Re-ran all basins (605) from delineation in test_basindelineation.ipynb with changes for the snap point, crop_dem function, and other fixes. Also refined the list to remove all basins with less than 35% intersection with the original GRDC shapefiles. All shapefiles used now are delineated by our code rather than the GRDC provided files. 



## Importing Packages

In [1]:
# !pip install pysheds netCDF4 fiona geopandas xarray pyshp sobel
# !pip install sobel

In [2]:
# IMPORT PACKAGES
import pysheds
from pysheds.grid import Grid
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.colors import ListedColormap, BoundaryNorm
import geopandas as gpd
import rasterio
import numpy as np
import pandas as pd
import fiona
import xarray as xr
import shapefile
import math
from scipy import stats
import os
from numpy import random
import netCDF4 as nc
from netCDF4 import Dataset
from shapely.geometry import Point, shape, box, mapping, Polygon, MultiPolygon
from shapely.vectorized import contains
from shapely.strtree import STRtree
import matplotlib.path as mpath
from scipy.ndimage import sobel
from rasterio.transform import rowcol
from rasterio.coords import BoundingBox
from rasterio.mask import mask
from rasterio.plot import show
from rasterio.features import shapes
print('Packages have been imported!')

Packages have been imported!


## Define Functions

### Crop Continent-Scale DEM

This function is not currently needed in this version since all the basins have been pre-delineated, but it is provided here for reference.

In [3]:
# def crop_dem(input_tif, output_tif, center_lon, center_lat,area, plot_scene = True):
#     """
#     Crop a GeoTIFF file around a specified coordinate (center_lon, center_lat), save the cropped image,
#     display the cropped area, and plot a scatter point at the center coordinates.
#     """

#     # # Define the path to the input and output TIFF files
#     # input_tiff = "path/to/your/input.tif"
#     # output_tif = "path/to/your/output_cropped.tif"
#     if area < 10000:
#       width = np.sqrt(area)*6 # width extending from center point to each edge of square
#     elif 10000< area < 50000:
#       width = np.sqrt(area)*5
#     elif 50000< area < 200000:
#       width = np.sqrt(area)*4
#     else:
#       width = np.sqrt(area)*3

#     width = int(width)
#     degrees = np.round(width/111.32,2)

#     xmin = center_lon - degrees
#     xmax = center_lon + degrees
#     ymin = center_lat - degrees
#     ymax = center_lat + degrees

#     # Define the bounding box coordinates (left, bottom, right, top)
#     bbox = (xmin, ymin, xmax, ymax)  # Replace with actual coordinates

#     # Open the source TIFF file
#     with rasterio.open(input_tif) as src:
#         print("Nodata value:", src.nodata)
#     # Convert the bounding box to a GeoJSON-style geometry for rasterio.mask.mask
#         bbox_geom = {
#             "type": "Polygon",
#             "coordinates": [[
#                 [bbox[0], bbox[1]],
#                 [bbox[0], bbox[3]],
#                 [bbox[2], bbox[3]],
#                 [bbox[2], bbox[1]],
#                 [bbox[0], bbox[1]]
#             ]]
#         }

#         # Crop the image using the bounding box geometry
#         out_image, out_transform = mask(src, [bbox_geom], crop=True)
#         out_meta = src.meta.copy()

#         # Update metadata to reflect the new cropped area
#         out_meta.update({
#             "driver": "GTiff",
#             "height": out_image.shape[1],
#             "width": out_image.shape[2],
#             "transform": out_transform,
#             'crs': src.crs  # Ensure the CRS remains the same
#         })

#         # # Save the cropped data to a new GeoTIFF file with updated profile
#         # profile = src.profile
#         # profile.update({
#         #     'height': height,
#         #     'width': width,
#         #     'transform': transform,
#         #     'crs': src.crs  # Ensure CRS is preserved
#         # })

#         with rasterio.open(output_tif, 'w', **out_meta) as dst:
#             dst.write(out_image)


#     # Plot the image using rasterio's show function
#     if plot_scene == True:
#         fig, ax = plt.subplots(figsize=(10, 10))

#         # Calculate the extent of the image in geographic coordinates
#         left = out_transform.c  # xmin
#         right = out_transform.c + out_transform.a * out_image.shape[2]  # xmax
#         top = out_transform.f  # ymax
#         bottom = out_transform.f + out_transform.e * out_image.shape[1]  # ymin
    
#         extent = (left, right, bottom, top)
#         show(out_image, transform=out_transform, extent=extent, ax=ax, cmap='gray')
#         # Set axis labels
#         ax.set_xlabel("Longitude")
#         ax.set_ylabel("Latitude")
#         # Add a scatter point at the center coordinate
#         plt.scatter(center_lon, center_lat, color='red', marker='o', s=100, label="Center Point")
#         plt.title("Cropped GeoTIFF Area with Center Point")
#         plt.title("Cropped Image")
#         plt.show()
#     return(out_image,bbox)

### Delineate Basin from DEM

These functions are not currently needed in this version since all the basins have been pre-delineated, but it is provided here for reference.

In [4]:
# def open_and_crop_basin(index, plot_scene = True):
#     print('\n1: Running crop method...')
#     # 1) Load the station
#     data = stations_df.iloc[index]

#     # Assign vars
#     number = data['grdc_no']
#     region = data['wmo_reg']
#     river = data['river']
#     name = data['station']
#     lat = data['lat']
#     lon = data['long']
#     area = data['area']
#     altitude = data['altitude']
#     print(f'  Station {number}\n   name: {name}\n   region: {region}\n   lon,lat: {lon},{lat}')

#     # 2) Assign region
#     SCRATCH_DIR = '/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/DATA/DEM/15s'
#     if region == 1:
#       dem = f'{SCRATCH_DIR}/hyd_af_dem_15s.tif'
#     elif region == 2:
#       dem = f'{SCRATCH_DIR}/hyd_as_dem_15s.tif'
#     elif region == 3:
#       dem = f'{SCRATCH_DIR}/hyd_sa_dem_15s.tif'
#     elif region == 4:
#       dem = f'{SCRATCH_DIR}/hyd_na_dem_15s.tif'
#     elif region == 5:
#       dem = f'{SCRATCH_DIR}/hyd_au_dem_15s.tif'
#     elif region == 6:
#       dem = f'{SCRATCH_DIR}/hyd_eu_dem_15s.tif'
#     print(f'   DEM: {dem}')
#     output_tif = f'/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/DATA/DEM/STATIONS/{number}_dem.tif' 

#     # 3) crop the region
#     output_dem, bbox = crop_dem(dem, output_tif, lon, lat,area, plot_scene)

#     return(bbox, lat, lon, area, number, output_tif, dem)

In [5]:
# def dem_condition_flowdir(output_tif, lon, lat, dem, number, plot_scene = True):
#     print('\n2a: Running delineation part 1...')
#     # GET THE FLOW MAP!
#     # Load the .tif file for the DEM
#     tif_input = output_tif
#     pour_point = [lon,lat]
#     print(f'   Target lon/lat:{pour_point}')
    
#     # Make dir for output tif...
#     new_folder_path = f'/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/DATA/DEM/STATIONS/{number}/'
#     os.makedirs(new_folder_path, exist_ok=True)
#     shp_name = new_folder_path + f'{number}.shp'

#     # Initialize a grid from the raster
#     grid = Grid.from_raster(tif_input, data_name='dem')
#     dem = grid.read_raster(tif_input)

#     # Condition dem
#     pit_filled_dem = grid.fill_pits(dem)
#     flooded_dem = grid.fill_depressions(pit_filled_dem)
#     inflated_dem = grid.resolve_flats(flooded_dem)

#     # Define the flow direction map (Dirmap)
#     dir_map = (64, 128, 1, 2, 4, 8, 16, 32)
#     f_dir = grid.flowdir(inflated_dem, dirmap=dir_map)  # returns a Raster object
#     print(np.nanmin(f_dir))
    
#     # Plot the flow direction map
#     if plot_scene == True:
#         fig = plt.figure(figsize=(8,6))
#         fig.patch.set_alpha(0)
        
#         plt.imshow(f_dir, extent=grid.extent, cmap='viridis', zorder=2)
#         plt.scatter(lon, lat, color='red', marker='o', s=100, label="Center Point")
#         boundaries = ([0] + sorted(list(dir_map)))
#         plt.colorbar(boundaries= boundaries,
#                      values=sorted(dir_map))
#         plt.xlabel('Longitude')
#         plt.ylabel('Latitude')
#         plt.title('Flow direction grid', size=14)
#         plt.grid(zorder=-1)
#         plt.tight_layout()
#     return(grid, dem, f_dir, dir_map, shp_name) 

In [6]:
# def snap_window(acc_raster, grid_raster, fdir, dirmap, longitude, latitude, window):
#     # Delin basin
#     row, col = rowcol(acc_raster.affine, longitude, latitude)
    
#     # Access the accumulation value
#     acc_pourpt = acc_raster.view()[row, col]
#     acc_maxwindow = int(acc_raster.max())
#     array = acc_raster.view()
    
#     # Define half-window size (9x9 = 4 px in each direction)
#     half_size = int((window - 1) / 2)
    
#     # Create a window around the pour-point to snap to
#     row_start = max(0, row - half_size)
#     row_end   = min(array.shape[0], row + half_size + 1)
#     col_start = max(0, col - half_size)
#     col_end   = min(array.shape[1], col + half_size + 1)
    
#     # Extract the window
#     window = array[row_start:row_end, col_start:col_end]

#     # Define the accumulation of the snapped point
#     snap_pt = np.max(window)
    
#     x_snapped, y_snapped = grid_raster.snap_to_mask(acc_raster >= snap_pt, (longitude, latitude))
#     catchment = grid_raster.catchment(x=x_snapped, y=y_snapped, fdir=fdir, dirmap=dirmap,
#                               xytype='coordinate')

#     return x_snapped, y_snapped, catchment, window, acc_pourpt, acc_maxwindow

In [7]:
# def snap_basin_acc(window_width, grid, dem, f_dir, dir_map, lon, lat, plot_scene = True):
#     print('\n2b: Running delineation part 2...')
#     # Get Accumulation
#     acc = grid.accumulation(f_dir, dirmap=dir_map)
#     acc[dem==0] = 0

#     # window_width is the number of pixels of the accumulation window centered on the pour point, width must be an odd number/integer
#     window_shape = [window_width,window_width]
    
#     x_snap, y_snap, catch, window, acc_atpt, acc_max = snap_window(acc, grid, f_dir, dir_map, lon, lat, window_width) 

#     # Clip the bounding box to the catchment
#     # grid.clip_to(catch)
#     catch_view = grid.view(catch, dtype=np.uint8)
#     n_cells = np.count_nonzero(catch_view)
#     print('number of cells: ', n_cells)
    
#     # Calculate grid size to meters
#     dx_m = np.cos(lat*(math.pi/180)) * 15 * 30.866666667 # m 
#     cell_area = dx_m ** 2  
    
#     catchment_area = n_cells * cell_area
#     catchment_area = catchment_area/1e6
#     print(f"Catchment area: {catchment_area:.2f} km²") 

    
#     # Exported metrics
#     print('\n  STATS-----')
#     print("   Flow accumulation at point: ", acc_atpt)
#     print("   Window shape:", window_shape) # window.shape)
#     max_acc_window = np.max(window)
#     print("   Max accumulation in 9x9: ", max_acc_window)# np.max(window)) 
#     mean_acc_window = np.mean(window)
#     print("   Avg accumulation in 9x9: ", mean_acc_window)#np.mean(window))
    
#     # Convert lon/lat to raster indices
#     row_snap, col_snap = rowcol(acc.affine, x_snap, y_snap)
#     # Access the accumulation value
#     acc_snapped = acc.view()[row_snap, col_snap]
#     acc_area = np.size(acc)
#     print(acc_area) # np.size(acc))
#     print(lon,lat)
#     print(x_snap, y_snap)
#     print("   Accumulation at snapped point: ", acc_snapped)
    
    
#     if plot_scene == True:
#         fig, ax = plt.subplots(figsize=(8,6))
#         fig.patch.set_alpha(0)
#         plt.grid('on', zorder=0)
#         im = ax.imshow(acc, extent=catch.extent, zorder=1,
#                         cmap='ocean',
#                         norm=colors.LogNorm(1, acc.max()),
#                         interpolation='bilinear')
#         plt.colorbar(im, ax=ax, label='Upstream Cells')
#         plt.imshow(np.where(catch_view,catch_view, np.nan), extent=grid.extent,zorder=2, cmap='Greys',alpha=0.3)
#         plt.scatter(lon,lat, color='yellow', s=10, marker='o', zorder=3)
#         plt.scatter(x_snap,y_snap, color='red', s=10, marker='o', zorder=3)
#         plt.xlabel('Longitude')
#         plt.ylabel('Latitude')
#         plt.tight_layout()
#         plt.show()
#     return(catch_view, catch, acc, x_snap, y_snap, acc_atpt, window_shape, max_acc_window, mean_acc_window, acc_area, acc_snapped)

In [8]:
# def extract_holes(geom):
#     """Return holes (interior rings) as polygons."""
#     holes = []
#     if geom.geom_type == "Polygon":
#         for interior in geom.interiors:
#             holes.append(Polygon(interior))
#     elif geom.geom_type == "MultiPolygon":
#         for poly in geom.geoms:
#             for interior in poly.interiors:
#                 holes.append(Polygon(interior))
#     return holes

In [9]:
# def get_utm_epsg(gdf): #need input grdc_shp

#     # check lat/lon
#     gdf_latlon = gdf.to_crs(4326)
#     centroid = gdf_latlon.geometry.centroid
#     lon = centroid.x.mean()
#     lat = centroid.y.mean()
#     print(f'calculated lat/lon, get_utm_espg fxn: {lon}, {lat}')
    
#     # get utm zone, calc espg
#     utm_zone = int((lon + 180) / 6) + 1
#     epsg_code = 32600 + utm_zone if lat >= 0 else 32700 + utm_zone
#     return epsg_code

In [10]:
# def compare_catchments(grid, catch_view, acc, catch, x_snap, y_snap, lon, lat, number, shp_name, area_GRDC):
#     print('\n3: Running comparison...')
#     shapes = grid.polygonize(catch_view)

#     # Specify schema
#     schema = {
#             'geometry': 'Polygon',
#             'properties': {'LABEL': 'float:16'}
#     }

#     # Write shapefile of the delineated catchment
#     shp_output = shp_name
#     with fiona.open(
#         shp_output,
#         'w',
#         driver='ESRI Shapefile',
#         crs=grid.crs.srs,
#         schema=schema
#     ) as c:
#         i = 0
#         for geom, value in shapes:
#             g = shape(geom)
    
#             # Extract interior rings (the regions you actually want)
#             holes = extract_holes(g)
    
#             # If no holes, just write the geometry itself
#             if not holes:
#                 rec = {
#                     "geometry": g.__geo_interface__,
#                     "properties": {"LABEL": str(value)},
#                     "id": str(i),
#                 }
#                 c.write(rec)
#                 i += 1
#             else:
#                 # Write each hole as its own polygon feature
#                 for h in holes:
#                     rec = {
#                         "geometry": h.__geo_interface__,
#                         "properties": {"LABEL": str(value)},
#                         "id": str(i),
#                     }
#                     c.write(rec)
#                     i += 1
                    
#     # Read the shapefiles
#     delin_shp = gpd.read_file(shp_output) # delineated shapefile
#     # shapefile1 = catch_view
#     grdc_shp = gpd.read_file(f'/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/DATA/SHAPEFILES/{number}/{number}.shp') # GRDC shapefile
    
#     # Step 3: Reproject to a projected CRS (e.g., UTM Zone 33N, replace EPSG:32633 as appropriate)
#     espg_code = get_utm_epsg(grdc_shp)
#     print(f' CALCED ESPG: {espg_code}')
#     projected_crs = grdc_shp.to_crs(epsg=espg_code)#32633)  # Use an appropriate EPSG for your region
#     delin_shp = delin_shp.to_crs(projected_crs.crs)
#     grdc_shp = grdc_shp.to_crs(projected_crs.crs)
    
#     # Calculate the intersection
#     intersection = gpd.overlay(delin_shp, grdc_shp, how='intersection')
#     #print(intersection)
    
#     # Calculate the area of the intersection (in m2)
#     intersection_area = intersection.dissolve().geometry.area
#     delin_area = delin_shp.dissolve().geometry.area
#     grdc_area = grdc_shp.dissolve().geometry.area
#     print('established area')
#     print(f'intersection area: {intersection_area.to_list()}')
#     print(f'delin area: {delin_area.to_list()}')
#     print(f'calculated grdc area: {grdc_area.to_list()}')
#     print(f'csv reported grdc area: {area_GRDC}')
#     inter_pcnt = ((intersection_area[0])/grdc_area[0])*100
#     print(inter_pcnt)
#     print(grdc_shp.crs)
#     print(delin_shp.crs)

#     # save plot
#     gdf_delin = gpd.read_file(shp_output)
#     gdf_grdc = gpd.read_file(f'/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/DATA/SHAPEFILES/{number}/{number}.shp')
    
#     fig, ax = plt.subplots(figsize=(8,6))
#     fig.patch.set_alpha(0)
    
#     # Flow accumulation background
#     im = ax.imshow(
#         acc,
#         extent=catch.extent,
#         zorder=1,
#         cmap="ocean",
#         norm=colors.LogNorm(1, acc.max()),
#         interpolation="bilinear"
#     )
    
#     # Colorbar
#     plt.colorbar(im, ax=ax, label="Upstream Cells")
    
#     # Overlay shapefiles instead of NaN mask
#     gdf_delin.boundary.plot(ax=ax, edgecolor="black", linewidth=1.5, zorder=2)
#     gdf_delin.plot(ax=ax, facecolor="red", edgecolor="red", alpha=0.5, zorder=2)
    
#     gdf_grdc.boundary.plot(ax=ax, edgecolor="black", linewidth=.6, zorder=2)
#     gdf_grdc.plot(ax=ax, facecolor="yellow", edgecolor="yellow", alpha=0.5, zorder=2)
    
#     # Snap point
#     ax.scatter(x_snap, y_snap, facecolor="red", edgecolor = 'black', s=90, linewidth = 2, marker="s", zorder=4, label = 'snapped')
#     ax.scatter(lon, lat, facecolor="yellow",edgecolor = 'black', linewidth = 2, s=50, marker="o", zorder=4, label = 'GRDC original')
    
#     # Labels, formatting
#     ax.set_xlabel("Longitude")
#     ax.set_ylabel("Latitude")
#     ax.grid(True, alpha = 0.5, zorder=0)

#     # Set limits
#     grdc_bounds = gdf_grdc.geometry.bounds
#     delin_bounds = gdf_delin.geometry.bounds

#     minx = min(grdc_bounds['minx'].min(), delin_bounds['minx'].min())
#     miny = min(grdc_bounds['miny'].min(), delin_bounds['miny'].min())
#     maxx = max(grdc_bounds['maxx'].max(), delin_bounds['maxx'].max())
#     maxy = max(grdc_bounds['maxy'].max(), delin_bounds['maxy'].max())
#     print(minx, maxx, miny, maxy)
#     # HATCHING OF INTERSECTION
#     intersection_crs_adjust = intersection.to_crs(gdf_grdc.crs)
    
#     if not intersection.empty:
#         intersection_crs_adjust.plot(
#             ax=ax,
#             facecolor="none",
#             edgecolor="black",
#             hatch="....",   
#             linewidth=0.0,
#             zorder=3
#             #label="Overlap Area"
#         )
#         intersection_crs_adjust.boundary.plot(ax=ax, color="black", linewidth=1, linestyle="--", zorder=3, label = 'Overlap')
    
#     #---
#     # ADD A LEGEND
#     plt.legend()
#     plt.tight_layout()
#     plt.title(f'Station ID {number}\nGRDC Index {n}')
#     plt.xlim(int(minx-1), int(maxx+1))
#     plt.ylim(int(miny-1), int(maxy+1))
#     #output_fn = f'/global/scratch/users/slsmit34/299/intersection_delin_plots/{number}_{n}.png'
#     output_fn = f'/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/RESULTS/DELINEATION_TEST/{number}_{n}.png'
#     plt.savefig(output_fn,dpi=300, bbox_inches='tight')
#     # plt.show()
#     return(gdf_grdc, grdc_area, intersection_area, inter_pcnt, output_fn, delin_area, delin_shp,grdc_shp)

### Load Streamflow GRDC Station Data

In [11]:
def get_stationdata(station_file):
  # Remove spaces in column names entirely
  df_Q = pd.read_csv(station_file, delimiter=';',encoding='utf-8',skiprows=38)
  # print(df_Q.head)

  df_Q.columns = df_Q.columns.str.replace(' ', '')
  df_Q['YYYY-MM-DD'] = df_Q['YYYY-MM-DD'].str.slice(0, 7)

  # Convert 'YYYY-MM-DD' column to datetime format
  df_Q['YYYY-MM-DD'] = pd.to_datetime(df_Q['YYYY-MM-DD'], format='%Y-%m')

  # Convert cutoff_date to datetime format
  start_date = pd.to_datetime('2018-08', format='%Y-%m')
  end_date = pd.to_datetime('2025-11', format='%Y-%m')

  # Filter the DataFrame to keep rows after '2018-08'
  df_filtered = df_Q[df_Q['YYYY-MM-DD'] >= start_date]
  df_filtered = df_filtered[df_filtered['YYYY-MM-DD'] <= end_date]

  # print(df_filtered.head)

  stream_gauge = df_filtered['Calculated'].values
  # print(len(stream_gauge))

  df_filtered['YYYY-MM-DD'] = df_filtered['YYYY-MM-DD'].dt.strftime('%Y-%m')
  dates = df_filtered['YYYY-MM-DD']
  dates = dates.reset_index()
  # print(dates)

  return stream_gauge, dates

### Calculate Water Pixel Count

In [12]:
def waterpx_count(shp_input, nc_input):

  # Open NetCDF file and extract variables
  with nc.Dataset(nc_input) as dataset:
      watermask = dataset.variables['watermask'][:]
      latitude = dataset.variables['lat'][:]
      longitude = dataset.variables['lon'][:]

  # Load and reproject the shapefile
  shp = gpd.read_file(shp_input).to_crs('EPSG:4326')
  minlon, minlat, maxlon, maxlat = shp.geometry.total_bounds

  # Limit the NetCDF data to the bounding box of the shapefile
  lat_mask = (latitude >= minlat) & (latitude <= maxlat)
  lon_mask = (longitude >= minlon) & (longitude <= maxlon)

  watermask = watermask[lat_mask, :][:, lon_mask]
  lat_filtered = latitude[lat_mask]
  lon_filtered = longitude[lon_mask]

  # Step 1: Create a grid of filtered points
  lon_grid, lat_grid = np.meshgrid(lon_filtered, lat_filtered)
  points = np.column_stack([lon_grid.ravel(), lat_grid.ravel()])

  # Flatten the watermask array to align with points
  watermask_flat = watermask.ravel()

  # Step 2: Load the shapefile and get combined geometry
  shapefile_geom = shp.geometry.unary_union  # Combine all geometries in the shapefile

  # Step 3: Identify points intersecting the shapefile
  intersects_mask = contains(shapefile_geom, points[:, 0], points[:, 1])

  # Step 4: Filter the points and watermask values
  filtered_points = points[intersects_mask]
  filtered_watermask = watermask_flat[intersects_mask]

  # Step 5: Create geometries for intersecting points
  geometries = [Point(lon, lat) for lon, lat in filtered_points]

  # Step 6: Create a GeoDataFrame
  gdf = gpd.GeoDataFrame({'watermask': filtered_watermask}, geometry=geometries, crs='EPSG:4326')


  # markercolormap2= colors.ListedColormap(['white', 'black','blue'])

  # # Assuming gdf and water_percent are defined, and markercolormap2 is valid
  # fig, ax = plt.subplots(1, 1, figsize=(10, 8))
  # gdf.plot(column='watermask', ax=ax, vmin=0, vmax=3, legend=True, markersize=5, cmap=markercolormap2)
  # ax.set_title(nc_input)
  # ax.set_xlabel('Longitude')
  # ax.set_ylabel('Latitude')
  # plt.show()

  # Step 1: Filter the GeoDataFrame where 'watermask' is equal to 2
  filtered_gdf = gdf[gdf['watermask'] == 1]

  water_pixels = len(filtered_gdf)
  total_pixels = len(gdf)

  water_percent = (water_pixels / total_pixels) * 100

  return water_pixels, water_percent, total_pixels

### Get Other Parameters (Slope, Precipitation, Land Cover, Soil Type)

In order for model: scaled streamflow, area, latitude, avg slope, max slope, precipitation, land cover, soil type

In [13]:
 def calculate_average_slope(dem_path, shapefile_path):
        # Load shapefile
        gdf = gpd.read_file(shapefile_path)
        # print("CRS of shapefile:", gdf.crs)
        shapes = gdf.geometry.values

        # Load DEM and mask it with the shapefile
        with rasterio.open(dem_path) as src:
            dem_data, transform = mask(src, shapes, crop=True)
            dem_data = dem_data[0].astype('float32')  # Convert to float

            if src.nodata is not None:
                dem_data[dem_data == src.nodata] = np.nan

            # Resolution in degrees
            xres_deg, yres_deg = src.res

            # Estimate latitude at center of masked area
            bounds = src.bounds
            center_lat = (bounds.top + bounds.bottom) / 2

            # Convert resolution from degrees to meters
            meters_per_degree_lat = 111320  # approx constant
            meters_per_degree_lon = 111320 * np.cos(np.radians(center_lat))

            xres_m = xres_deg * meters_per_degree_lon
            yres_m = yres_deg * meters_per_degree_lat

        # Compute slope using Sobel filters (rise/run)
        dzdx = sobel(dem_data, axis=1, mode='nearest') / (8 * xres_m)
        dzdy = sobel(dem_data, axis=0, mode='nearest') / (8 * yres_m)
    
        # Slope in degrees
        slope_rad = np.arctan(np.sqrt(dzdx**2 + dzdy**2))
        slope_deg = np.degrees(slope_rad)
    
        # Average slope
        avg_slope = np.nanmean(slope_deg)
        max_slope = np.nanmax(slope_deg)
        return avg_slope, max_slope

In [14]:
def calc_avg_precip(shp_input, date_input):
    precip_folder = '/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/DATA/PRECIP/RESAMPLED/'
    filename = f"IMERG-Final.CLIM.2001-2022.{date_input}.V07B.nc4"
    
    nc_input = os.path.join(precip_folder, filename)
    
    # Open NetCDF file and extract variables
    with nc.Dataset(nc_input) as dataset:
        precip = dataset.variables['precipitation'][:]  # shape (time, lat, lon) or (lat, lon)
        # print(precip.shape)
        latitude = dataset.variables['lat'][:]  # shape (lat,)
        longitude = dataset.variables['lon'][:]  # shape (lon,)
    
    # Load and reproject the shapefile
    shp = gpd.read_file(shp_input).to_crs('EPSG:4326')
    minlon, minlat, maxlon, maxlat = shp.geometry.total_bounds
    
    # Limit the NetCDF data to the bounding box of the shapefile
    lat_mask = (latitude >= minlat) & (latitude <= maxlat)
    lon_mask = (longitude >= minlon) & (longitude <= maxlon)
    
    # Filter latitude and longitude based on the mask
    lat_filtered = latitude[lat_mask]
    lon_filtered = longitude[lon_mask]
    # print(lat_filtered,lon_filtered)
    
    # Step 1: Create a grid of filtered points
    lon_grid, lat_grid = np.meshgrid(lon_filtered, lat_filtered)
    points = np.column_stack([lon_grid.ravel(), lat_grid.ravel()])
    
    # Check the dimensionality of precip
    if precip.ndim == 2:  # If it's (lat, lon)
        precip_filtered = precip[:,lat_mask][lon_mask,:]  # Apply both lat and lon masks
    elif precip.ndim == 3:  # If it's (time, lat, lon)
        precip_filtered = precip[:, lat_mask, :][:, :, lon_mask]
    
    # Flatten the filtered precipitation array (time dimension included if 3D)
    precip_flat = precip_filtered.ravel()
    
    # Step 2: Load the shapefile and get combined geometry
    shapefile_geom = shp.geometry.unary_union  # Combine all geometries in the shapefile
    
    # Step 3: Identify points intersecting the shapefile
    intersects_mask = contains(shapefile_geom, points[:, 0], points[:, 1])
    
    # Step 4: Filter the points and precipitation values
    filtered_points = points[intersects_mask]
    filtered_precip = precip_flat[intersects_mask]
    
    avg = np.mean(filtered_precip)

    return avg

In [15]:
def landcover_percentage(shp_input):

    landcover_map = '/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/DATA/landcover_mosaic.tif'
    gdf = gpd.read_file(shp_input)

    # File paths
    geotiff_path = landcover_map
    
    # Open the GeoTIFF
    with rasterio.open(geotiff_path) as src:
        # Reproject shapefile to match raster CRS
        gdf = gdf.to_crs(src.crs)
    
        # Convert shapefile geometry to GeoJSON-like format
        shapes = [mapping(geom) for geom in gdf.geometry]
    
        # Mask raster with shapefile
        out_image, out_transform = rasterio.mask.mask(src, shapes, crop=True, filled=True)
        out_meta = src.meta.copy()
    
        # Update metadata
        out_meta.update({
            "driver": "GTiff",
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform
        })

    # Masked array: out_image is (1, height, width) with nodata masked
    masked_array = np.ma.masked_array(out_image, out_image == src.nodata)
    
    # Alternatively, use the mask from the output:
    valid_pixels = masked_array.compressed()

    # Count occurrences of each band value (1-20)
    unique, counts = np.unique(valid_pixels, return_counts=True)
    pixel_counts = dict(zip(unique, counts))

    # Compute percentage of total valid pixels
    total_pixels = valid_pixels.size
    percentages = {i: (pixel_counts[i] / total_pixels * 100) if i in pixel_counts else 0 for i in range(1, 21)}

    # Convert percentages to a DataFrame row
    landcover_row = pd.DataFrame([percentages])
    
    landcover_row.columns = [f"LC_{col}" for col in landcover_row.columns]

    landcover_row["number"] = number  # Add 'number' as a separate column
    landcover_row.set_index("number", inplace=True)  # Set 'number' as the index

    return landcover_row

In [16]:
def soiltype_percentage(shp_input):

    nc_file = '/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/DATA/GLDASp5_soiltexture_025d.nc4'
    
    # Load the NetCDF data
    with nc.Dataset(nc_file) as dataset:
        soil = dataset.variables['GLDAS_soiltex'][:]  # Soil data (masked)
        latitude = dataset.variables['lat'][:]  # Latitude values
        longitude = dataset.variables['lon'][:]  # Longitude values
        soil_masked = dataset.variables['GLDAS_soiltex'][:]
        soil_masked = np.ma.masked_equal(soil_masked, -9999)  # Mask invalid values (assuming -9999 is used for missing data)

    # Remove the singleton time dimension (if present)
    soil = soil_masked.squeeze()  # Removes the first dimension if it's size 1

    # Check the shape after squeezing
    # print(f"Shape after squeezing: {soil.shape}")

    # Convert the masked array to a regular numpy array with NaN for masked values
    soil = soil.filled(np.nan)

    # # Check if the entire soil data is NaN
    # if np.all(np.isnan(soil)):
    #     print("Warning: All soil data is NaN after masking. Please check the data source or bounds.")
    # else:
    #     print(f"Data contains valid values. Shape: {soil.shape}")

    # Create an xarray Dataset for easy interpolation
    data_set = xr.Dataset({"soil": (["lat", "lon"], soil)},
                    coords={"lat": latitude, "lon": longitude})

    # Load the shapefile and reproject it to EPSG:4326 if needed
    shp = gpd.read_file(shp_input).to_crs('EPSG:4326')
    minlon, minlat, maxlon, maxlat = shp.geometry.total_bounds

    # Crop the data to the bounds of the shapefile
    lat_mask = (latitude >= minlat) & (latitude <= maxlat)
    lon_mask = (longitude >= minlon) & (longitude <= maxlon)
    cropped_soil = soil[lat_mask, :][:, lon_mask]

    # Check if the cropped data is empty or full of NaNs
    if np.all(np.isnan(cropped_soil)):
        print("Warning: Cropped soil data is empty or full of NaNs. Please check your shapefile bounds or data.")
    else:
        print(f"Cropped data has valid values. Shape: {cropped_soil.shape}")

    # Create new latitude and longitude arrays for the desired resolution
    dx_new = 0.1  # Desired resolution
    newlon = np.arange(minlon, maxlon, dx_new)
    newlat = np.arange(minlat, maxlat, dx_new)

    # Interpolate the data to the new grid
    data_set_interp = data_set.interp(lat=newlat, lon=newlon)

    # Extract the 'soil' data as a NumPy array from the xarray Dataset
    soil = data_set_interp['soil'].values

    # Get interpolated lat/lon arrays
    lat_vals = data_set_interp['lat'].values
    lon_vals = data_set_interp['lon'].values

    # Create meshgrid of coordinates
    lon_grid, lat_grid = np.meshgrid(lon_vals, lat_vals)

    # Flatten the grids for masking
    lat_flat = lat_grid.flatten()
    lon_flat = lon_grid.flatten()
    soil_flat = soil.flatten()

    # Get the first (or union) geometry from the shapefile
    geom = shp.unary_union  # For multi-polygons

    # Mask: only points within the shapefile geometry
    mask = contains(geom, lon_flat, lat_flat)

    # Apply mask to soil data
    soil_in_geom = soil_flat[mask]

    # Filter out NaNs and values outside 1–16
    soil_in_geom = soil_in_geom[~np.isnan(soil_in_geom)]
    soil_in_geom = soil_in_geom[(soil_in_geom >= 1) & (soil_in_geom <= 16)]

    # Ensure integer values for counting
    soil_in_geom = soil_in_geom.astype(int)

    # Count valid values
    unique, counts = np.unique(soil_in_geom, return_counts=True)
    soil_counts = dict(zip(unique, counts))

    # Total number of valid soil class pixels in the shape
    total_points = np.sum(counts)

    # Calculate percentage for each class (1–16)
    percentages = {i: (soil_counts.get(i, 0) / total_points * 100) for i in range(1, 17)}

    # Debug: check sum of percentages
    # print(f"Sum of percentages: {sum(percentages.values()):.2f}%")

    # Display the results as a pandas Series for better readability
    percentages_df = pd.Series(percentages).sort_index()

    percentages_df = percentages_df.T

    most_common = percentages_df.idxmax()

    # print(most_common)
    # Convert percentages to a DataFrame row
    soil_row = pd.DataFrame([percentages])
    soil_row.columns = [f"ST_{col}" for col in soil_row.columns]
    soil_row["number"] = number  # Add 'number' as a separate column
    soil_row.set_index("number", inplace=True)  # Set 'number' as the index

    return soil_row

## Define Filepaths and Open Station Masterfile


In [17]:
# DEFINE INPUTS FOR FUNCTION
masterlist = '/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/DATA/GRDC_Stations_UPDATED.csv'

In [18]:
stations_df = pd.read_csv(masterlist)
# print(stations_df)
station_num = stations_df['grdc_no']
# station_num

final_result = pd.DataFrame()
print(f"Loaded {len(station_num)} stations.") # as of Nov 2025: should be 578 stations total when loading all

Loaded 578 stations.


## Loop Through All Stations

In [21]:
import warnings
warnings.filterwarnings("ignore")

for s in range(10,11):
    
    station = station_num[s]
    print("Running station ", station)
    
    data = stations_df.iloc[s]

    number = station
    region = data['wmo_reg']
    river = data['river']
    name = data['station']
    lat = data['lat']
    lon = data['long']
    area = data['area_delin']
    altitude = data['altitude']
    
    q_file = f'/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/DATA/STATIONS/{number}_Q_Month.txt' # GRDC monthly streamgauge readings as .txt files
    shp_file = f'/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/DATA/DEM/STATIONS/{station}/{station}.shp' # all station delineated shapefiles saved in subfolders named by station number
    dem_file = f'/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/DATA/DEM/STATIONS/{station}_dem.tif' # cropped .tif files for 15 arc-second DEM saved in subfolders named by station number
    
    results_df = pd.DataFrame(stations_df.iloc[[s]])
    
    stream_gauge, dates = get_stationdata(q_file)
    
    # assign directory
    directory = "/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CYGNSS/Data/MONTHLY_WATERMASK/"
    file_list = [f for f in os.listdir(directory) if os.path.isfile(os.path.join(directory, f))]

    # Calculate water pixel count
    ncdf_list = []
    water_px = np.zeros(len(dates))
    water_pcnt = np.zeros(len(dates)) 
    water_area = np.zeros(len(dates))
    tot_px = np.zeros(len(dates))
    
    for i in range(0,len(dates)):
        date = dates['YYYY-MM-DD'][i]
        ncdf_name = f'/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CYGNSS/Data/MONTHLY_WATERMASK/cyg.ddmi.{date}.l3.uc-berkeley-watermask-monthly.a31.d32.nc'
        ncdf_list.append(ncdf_name)
    
    for f, filename in enumerate(ncdf_list):
      if os.path.isfile(filename):
        pixel_count, pixel_percent, total_pixels = waterpx_count(shp_file, filename)
        water_px[f] = pixel_count
        water_pcnt[f] = pixel_percent
        tot_px[f] = total_pixels
        water_area[f] = ((pixel_percent*area)/100)

    print("water px count calculated")
    # Save Individual Results
    new_folder_path = f'/global/scratch/users/arvalcarcel/CSMUB/RESULTS/{number}/'
    os.makedirs(new_folder_path, exist_ok=True)

    # Get Other Parameters: Precipitation Climatology
    precip = np.zeros(len(dates))
    date_inputs = dates['YYYY-MM-DD']
    for index, date in enumerate(date_inputs):
        date_input = date[-2:]
        # Compute average precipitation for this date and shapefile
        precip[index] = calc_avg_precip(shp_file, date_input)
    
    df_final = pd.DataFrame({'Date': dates['YYYY-MM-DD'], 'Q': stream_gauge,'SWE': water_pcnt, 'SWE_scaled': water_area, 'P': precip})
    print("Individual results:")
    print(df_final)
    
    # Calculate Statistics
    slope, intercept, r, p, se = stats.linregress(stream_gauge, water_pcnt)
    r2 = r**2
    results_df['r2'] = r2

    # Get Other Parameters: Slope, Landcover, Soil Type
    average_slope, maximum_slope = calculate_average_slope(dem_file, shp_file)
    results_df['avg_slope'] = average_slope
    results_df['max_slope'] = maximum_slope
    landcover_classes = landcover_percentage(shp_file).reset_index(drop=True)
    print(landcover_classes)
    soil_types = soiltype_percentage(shp_file).reset_index(drop=True)
    print(soil_types)
    results_df = results_df.reset_index(drop=True)

    # Concatenate horizontally (side by side)
    combined_row = pd.concat([results_df, landcover_classes, soil_types], axis=1)
    
    # Append that single row to the final results
    final_result = pd.concat([final_result, combined_row], ignore_index=True)
    print("Final results:")
    print(final_result.head())

    
# final_result.to_csv(final_path, index=False)

Running station  1159305
water px count calculated
Individual results:
       Date      Q       SWE  SWE_scaled         P
0   2018-08  0.011  1.450701  124.944583  0.446584
1   2018-09  0.007  1.991640  171.534089  0.461480
2   2018-10  0.005  1.831817  157.769007  0.912444
3   2018-11  0.000  1.081879   93.179011  1.103482
4   2018-12  0.000  1.807229  155.651303  1.603053
..      ...    ...       ...         ...       ...
60  2023-08  0.363  2.458815  211.770480  0.446584
61  2023-09  0.216  1.413819  121.768026  0.461480
62  2023-10  0.166  1.278584  110.120650  0.912444
63  2023-11  0.096  1.352348  116.473764  1.103482
64  2023-12  1.225  1.659700  142.945074  1.603053

[65 rows x 5 columns]
   LC_1       LC_2      LC_3      LC_4      LC_5      LC_6      LC_7  \
0     0  89.142437  9.385351  0.809717  0.331248  0.134953  0.098147   

       LC_8      LC_9     LC_10     LC_11  LC_12  LC_13  LC_14  LC_15  LC_16  \
0  0.036805  0.012268  0.024537  0.024537      0      0      0      0

In [ ]:
# # for i in range(0):
# # for station in station_num[0]:
# i = 0
# station = station_num[i]
# data = stations_df.iloc[i]
# # print(data)
# number = station
# region = data['wmo_reg']
# river = data['river']
# name = data['station']
# lat = data['lat']
# lon = data['long']
# area = data['area_delin']
# altitude = data['altitude']

# print(number)
# print(region)
# print(name)
# print(lon,lat)

# q_file = f'/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/DATA/STATIONS/{number}_Q_Month.txt' # GRDC monthly streamgauge readings as .txt files
# shp_file = f'/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/DATA/DEM/STATIONS/{station}/{station}.shp' # all station delineated shapefiles saved in subfolders named by station number
# dem_file = f'/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/DATA/DEM/STATIONS/{station}/{station}_dem.tif' # cropped .tif files for 15 arc-second DEM saved in subfolders named by station number

# results_df = pd.DataFrame(stations_df.iloc[[i]])
# print(results_df)

### Get Streamflow Data

In [20]:
stream_gauge, dates = get_stationdata(q_file)

In [21]:
dates['YYYY-MM-DD']

0     2018-08
1     2018-09
2     2018-10
3     2018-11
4     2018-12
       ...   
74    2024-10
75    2024-11
76    2024-12
77    2025-01
78    2025-02
Name: YYYY-MM-DD, Length: 79, dtype: object

### Calculate Water Pixel Percent

In [22]:
# assign directory
directory = "/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CYGNSS/Data/MONTHLY_WATERMASK/"
file_list = [f for f in os.listdir(directory) if os.path.isfile(os.path.join(directory, f))]

In [23]:
dates

,index,YYYY-MM-DD
0,993,2018-08
1,994,2018-09
2,995,2018-10
3,996,2018-11
4,997,2018-12
...,...,...
74,1067,2024-10
75,1068,2024-11
76,1069,2024-12
77,1070,2025-01


In [24]:
# Create empty numpy arrays with the correct size
# num_files = len(file_list)
ncdf_list = []
water_px = np.zeros(len(dates))
water_pcnt = np.zeros(len(dates)) 
water_area = np.zeros(len(dates))
tot_px = np.zeros(len(dates))

In [25]:
for i in range(0,len(dates)):
    date = dates['YYYY-MM-DD'][i]
    ncdf_name = f'/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CYGNSS/Data/MONTHLY_WATERMASK/cyg.ddmi.{date}.l3.uc-berkeley-watermask-monthly.a31.d32.nc'
    ncdf_list.append(ncdf_name)

In [26]:
# ncdf_list

In [27]:
for f, filename in enumerate(ncdf_list):
  if os.path.isfile(filename):
    pixel_count, pixel_percent, total_pixels = waterpx_count(shp_file, filename)
    water_px[f] = pixel_count
    water_pcnt[f] = pixel_percent
    tot_px[f] = total_pixels
    water_area[f] = ((pixel_percent*area)/100)

/tmp/ipykernel_2598099/3313434207.py:29: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  shapefile_geom = shp.geometry.unary_union  # Combine all geometries in the shapefile
/tmp/ipykernel_2598099/3313434207.py:29: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  shapefile_geom = shp.geometry.unary_union  # Combine all geometries in the shapefile
/tmp/ipykernel_2598099/3313434207.py:29: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  shapefile_geom = shp.geometry.unary_union  # Combine all geometries in the shapefile
/tmp/ipykernel_2598099/3313434207.py:29: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  shapefile_geom = shp.geometry.unary_union  # Combine all geometries in the shapefile
/tmp/ipykernel_2598099/3313434207.py:29: DeprecationWarning: The 'unary_union' attri

In [28]:
print(water_px)

[16615. 14607. 15722. 13001. 14194. 14596. 16453. 16335. 21903. 20532.
 16771. 16369. 14802. 14352. 14897. 13988. 15369. 17302. 16522. 17615.
 19854. 17509. 15066. 15015. 13305. 14416. 12690. 16604. 16926. 18985.
 17844. 16728. 16031. 17244. 16869. 15000. 15671. 14569. 14525. 14937.
 20884. 21108. 20233. 20401. 25161. 23586. 25081. 22663. 18977. 16360.
 16215. 22474. 20022. 16823. 19556. 17878. 17245. 18241. 19022. 18207.
 16377. 14784. 14757. 14272. 15401. 16699. 15419. 14364. 19067. 14932.
 17975. 15566. 15247. 15119. 14025. 13618. 13039. 14555. 16004.]


### Save Individual Results

In [29]:
# Define the path for the new folder
new_folder_path = f'/global/scratch/users/arvalcarcel/CSMUB/RESULTS/{number}/'

# Create the folder
os.makedirs(new_folder_path, exist_ok=True)

In [30]:
df_final = pd.DataFrame({'Date': dates['YYYY-MM-DD'], 'Q': stream_gauge,'SWE': water_pcnt, 'SWE_scaled': water_area})
print(df_final)

       Date        Q       SWE    SWE_scaled
0   2018-08   24.526  2.084813  18220.936879
1   2018-09   31.372  1.832854  16018.851941
2   2018-10   19.572  1.972761  17241.623209
3   2018-11    7.349  1.631336  14257.622652
4   2018-12   13.824  1.781031  15565.933076
..      ...      ...       ...           ...
74  2024-10   11.830  1.759826  15380.598239
75  2024-11    5.099  1.708756  14934.259309
76  2024-12    6.316  1.636104  14299.295574
77  2025-01    8.765  1.826329  15961.825837
78  2025-02 -999.000  2.008146  17550.880158

[79 rows x 4 columns]


### Get Other Parameters: Precipitation

In [53]:
# NEED TO CHANGE THIS!!!
precip = np.zeros(len(dates))
date_inputs = dates['YYYY-MM-DD']
for index, date in enumerate(date_inputs):
    # print(date)
    date_input = date[-2:]
    # print(date_input)
    
    # Compute average precipitation for this date and shapefile
    precip[index] = calc_avg_precip(shp_file, date_input)

df_final['P'] = precip # in [mm]

# final_path = '/global/scratch/users/arvalcarcel/CSMUB/RESULTS/ALL_STATIONS_ALL_MONTHS_PRECIP.csv'
# stations_df.to_csv(final_path, index=False)

/tmp/ipykernel_2598099/1718360571.py:41: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  shapefile_geom = shp.geometry.unary_union  # Combine all geometries in the shapefile
/tmp/ipykernel_2598099/1718360571.py:41: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  shapefile_geom = shp.geometry.unary_union  # Combine all geometries in the shapefile
/tmp/ipykernel_2598099/1718360571.py:41: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  shapefile_geom = shp.geometry.unary_union  # Combine all geometries in the shapefile
/tmp/ipykernel_2598099/1718360571.py:41: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  shapefile_geom = shp.geometry.unary_union  # Combine all geometries in the shapefile
/tmp/ipykernel_2598099/1718360571.py:41: DeprecationWarning: The 'unary_union' attri

In [54]:
print(df_final)
# df_final.to_csv(f"/global/scratch/users/arvalcarcel/CSMUB/RESULTS/CSV/{number}.csv")

       Date        Q       SWE    SWE_scaled             P
0   2018-08   24.526  2.084813  18220.936879  2.298420e+05
1   2018-09   31.372  1.832854  16018.851941  2.988707e+05
2   2018-10   19.572  1.972761  17241.623209  8.093591e+05
3   2018-11    7.349  1.631336  14257.622652  1.300287e+06
4   2018-12   13.824  1.781031  15565.933076  1.728379e+06
..      ...      ...       ...           ...           ...
74  2024-10   11.830  1.759826  15380.598239  8.093591e+05
75  2024-11    5.099  1.708756  14934.259309  1.300287e+06
76  2024-12    6.316  1.636104  14299.295574  1.728379e+06
77  2025-01    8.765  1.826329  15961.825837  2.337996e+06
78  2025-02 -999.000  2.008146  17550.880158  2.292116e+06

[79 rows x 5 columns]


### Add Statistics

In [32]:
slope, intercept, r, p, se = stats.linregress(stream_gauge, water_pcnt)

In [33]:
r2 = r**2
results_df['r2'] = r2

In [34]:
results_df

,grdc_no,wmo_reg,sub_reg,river,station,country,lat,long,area,area_delin,...,m_end,m_yrs,m_miss,t_start,t_end,t_yrs,lta_discharge,r_volume_yr,r_height_yr,r2
0,1159100,1,1591,ORANGE RIVER,VIOOLSDRIF (27811003),ZA,-28.75799,17.721591,866486.0,873984.263,...,2022.0,88.0,5.379443,1935,2025,91,249.29,7.86160944,9.072979183,0.319072


In [35]:
# fig, ax1 = plt.subplots()

# ax1.plot(stream_gauge, color='b', label='Streamflow')
# ax1.set_xlabel('Time')
# ax1.set_ylabel('Streamflow [mg/s]', color='b')
# ax1.set_title(f'{name} Streamflow vs Watermask Timeseries, R2 = {r2}')
# # ax1.set_xticklabels(['2017-10','2018-08','2019-06','2020-04','2021-02','2021-12','2022-10','2023-08','2024-04'])
# # ax1.set_xticklabels(dates)
# ax2 = ax1.twinx()
# ax2.plot(water_pcnt, color='r', label='Watermask Px Percentage')
# ax2.set_ylabel('Watermask Px Percent', color='r')

# # Adding the legend inside the plot
# lines, labels = ax1.get_legend_handles_labels()
# lines2, labels2 = ax2.get_legend_handles_labels()
# ax1.legend(lines + lines2, labels + labels2, loc='upper center', bbox_to_anchor=(0.5, -0.1), ncol=2)
# plt.savefig(f"/global/scratch/users/arvalcarcel/CSMUB/RESULTS/TIMESERIES/{number}.png",bbox_inches = "tight")
# plt.show()

### Get Other Parameters: Avg and Max Slope

In [36]:
# Example usage
average_slope, maximum_slope = calculate_average_slope(dem_file, shp_file)
# print(f"Average slope within the area: {average_slope:.2f} degrees")

# Merge total_df with landcover_df on GRDC_No
# Store the average slope in the DataFrame
results_df['avg_slope'] = average_slope
results_df['max_slope'] = maximum_slope

### Get Other Parameters: Landcover Classification

In [38]:
landcover_classes = soiltype_percentage(shp_file)
print(landcover_classes)
# Concatenate columns side-by-side
results_df = pd.concat([results_df, landcover_classes], axis=1)

Cropped data has valid values. Shape: (44, 55)
           1         2          3         4         5          6         7  \
number                                                                       
1159100  0.0  8.116924  47.446995  2.609459  6.147284  24.074771  1.204366   

                8         9        10        11        12   13   14   15   16  
number                                                                         
1159100  1.480366  5.331828  0.401455  1.016184  2.170368  0.0  0.0  0.0  0.0  


/tmp/ipykernel_2598099/761837109.py:71: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geom = shp.unary_union  # For multi-polygons


### Get Other Parameters: Soil Type

In [39]:
soil_types = landcover_percentage(shp_file)
print(soil_types)
# Concatenate columns side-by-side
results_df = pd.concat([results_df, soil_types], axis=1)

                1          2          3         4         5         6  \
number                                                                  
1159100  6.118091  60.465347  17.852514  3.575547  1.867761  1.407659   

                7         8         9        10        11        12        13  \
number                                                                          
1159100  1.168386  0.986704  0.848811  0.784319  0.724093  0.676791  0.633127   

               14        15        16        17        18  19  20  
number                                                             
1159100  0.624218  0.631119  0.745047  0.804645  0.085822   0   0  


### Save Results

In [40]:
print(results_df.head())

           grdc_no  wmo_reg  sub_reg         river                station  \
0        1159100.0      1.0   1591.0  ORANGE RIVER  VIOOLSDRIF (27811003)   
1159100        NaN      NaN      NaN           NaN                    NaN   

        country       lat       long      area  area_delin  ...        11  \
0            ZA -28.75799  17.721591  866486.0  873984.263  ...       NaN   
1159100     NaN       NaN        NaN       NaN         NaN  ...  0.724093   

               12        13        14        15        16        17        18  \
0             NaN       NaN       NaN       NaN       NaN       NaN       NaN   
1159100  0.676791  0.633127  0.624218  0.631119  0.745047  0.804645  0.085822   

          19   20  
0        NaN  NaN  
1159100  0.0  0.0  

[2 rows x 65 columns]


In [41]:
final_result = pd.concat([final_result, results_df], ignore_index=True)
print(final_result)
# final_result.to_csv(final_path, index=False)

     grdc_no  wmo_reg  sub_reg         river                station country  \
0  1159100.0      1.0   1591.0  ORANGE RIVER  VIOOLSDRIF (27811003)      ZA   
1        NaN      NaN      NaN           NaN                    NaN     NaN   

        lat       long      area  area_delin  ...        11        12  \
0 -28.75799  17.721591  866486.0  873984.263  ...       NaN       NaN   
1       NaN        NaN       NaN         NaN  ...  0.724093  0.676791   

         13        14        15        16        17        18   19   20  
0       NaN       NaN       NaN       NaN       NaN       NaN  NaN  NaN  
1  0.633127  0.624218  0.631119  0.745047  0.804645  0.085822  0.0  0.0  

[2 rows x 65 columns]
